# Preprocessing walkthrough: one run (S001 R04)

Each step of `preprocess.py`, applied to a single run.

In [1]:
import mne
from mne.datasets import eegbci

subject, run = 1, 4   # run 4 = imagined left vs right fist

fname = eegbci.load_data(subject, [run])[0]
raw = mne.io.read_raw_edf(fname, preload=True, verbose=False)

# standard channel names + 10-05 montage
eegbci.standardize(raw)
raw.set_montage(mne.channels.make_standard_montage("standard_1005"))

print(raw)

<RawEDF | S001R04.edf, 64 x 20000 (125.0 s), ~9.8 MiB, data loaded>


In [2]:
# high-pass 1 Hz (before ICA)
raw.filter(1.0, None, fir_design="firwin", verbose=False)

print(f"Highpass: {raw.info['highpass']} Hz")
print(f"Lowpass:  {raw.info['lowpass']} Hz")

Highpass: 1.0 Hz
Lowpass:  80.0 Hz


In [3]:
# average reference
raw.set_eeg_reference("average", projection=False, verbose=False)

print(f"Reference applied: {raw.info['custom_ref_applied']}")

Reference applied: 1 (FIFFV_MNE_CUSTOM_REF_ON)


In [4]:
from autoreject import Ransac

# RANSAC bad-channel detection on temporary epochs
events, _ = mne.events_from_annotations(raw, verbose=False)
epochs_tmp = mne.Epochs(raw, events, tmin=0.0, tmax=4.0,
                        baseline=None, preload=True, verbose=False)

ransac = Ransac(verbose=False, n_jobs=1)
ransac.fit(epochs_tmp)

raw.info["bads"] = ransac.bad_chs_
if ransac.bad_chs_:
    raw.interpolate_bads(reset_bads=True, verbose=False)

print(f"Bad channels: {ransac.bad_chs_}")

Bad channels: []


In [5]:
# ICA (Picard, n_components = data rank)
from mne.preprocessing import ICA

rank = mne.compute_rank(raw, verbose=False)["eeg"]

ica = ICA(n_components=rank, method="picard", max_iter=1000, random_state=42,
          fit_params=dict(ortho=False, extended=True))
ica.fit(raw, verbose=False)

print(f"ICA fitted: {ica.n_components_} components")

ICA fitted: 63 components


In [6]:
# ICLabel: classify each component
from mne_icalabel import label_components

ic_labels = label_components(raw, ica, method="iclabel")

for i, (label, prob) in enumerate(zip(ic_labels["labels"], ic_labels["y_pred_proba"])):
    print(f"IC{i:02d}: {label} ({prob:.2f})")

IC00: eye blink (0.98)
IC01: brain (0.33)
IC02: brain (0.85)
IC03: brain (0.75)
IC04: brain (0.95)
IC05: brain (0.88)
IC06: brain (0.48)
IC07: brain (0.99)
IC08: other (0.63)
IC09: line noise (0.43)
IC10: brain (0.93)
IC11: other (0.61)
IC12: brain (0.95)
IC13: brain (0.96)
IC14: brain (0.79)
IC15: brain (0.59)
IC16: other (0.50)
IC17: brain (0.89)
IC18: brain (0.97)
IC19: muscle artifact (0.51)
IC20: brain (0.93)
IC21: other (0.73)
IC22: brain (0.87)
IC23: other (0.73)
IC24: other (0.50)
IC25: brain (0.97)
IC26: brain (0.96)
IC27: brain (0.71)
IC28: brain (0.34)
IC29: line noise (0.34)
IC30: brain (0.91)
IC31: brain (0.38)
IC32: brain (0.70)
IC33: other (0.48)
IC34: other (0.99)
IC35: other (0.90)
IC36: brain (0.74)
IC37: line noise (0.86)
IC38: muscle artifact (0.83)
IC39: line noise (0.45)
IC40: brain (0.66)
IC41: eye blink (0.33)
IC42: other (0.54)
IC43: other (0.57)
IC44: other (0.58)
IC45: other (0.85)
IC46: other (0.60)
IC47: brain (0.64)
IC48: muscle artifact (0.98)
IC49: muscl

In [7]:
# remove non-brain components with p > 0.8
threshold = 0.8
exclude = [i for i, (label, prob) in enumerate(zip(ic_labels["labels"], ic_labels["y_pred_proba"]))
           if label not in ("brain", "other") and prob > threshold]

print(f"Excluding {len(exclude)} components: {exclude}")
for i in exclude:
    print(f"  IC{i:02d}: {ic_labels['labels'][i]} ({ic_labels['y_pred_proba'][i]:.2f})")

ica.exclude = exclude
ica.apply(raw, verbose=False)

Excluding 5 components: [0, 37, 38, 48, 51]
  IC00: eye blink (0.98)
  IC37: line noise (0.86)
  IC38: muscle artifact (0.83)
  IC48: muscle artifact (0.98)
  IC51: eye blink (0.88)


In [8]:
# low-pass 40 Hz (after ICA, so ICLabel sees muscle activity)
raw.filter(None, 40.0, fir_design="firwin", verbose=False)
print(f"Highpass: {raw.info['highpass']} Hz | Lowpass: {raw.info['lowpass']} Hz")

Highpass: 1.0 Hz | Lowpass: 40.0 Hz


In [9]:
# resample to 128 Hz
raw.resample(128.0, verbose=False)
print(f"New sampling rate: {raw.info['sfreq']} Hz")

New sampling rate: 128.0 Hz


In [10]:
# epochs -0.5 to 4 s around T1/T2
events, event_map = mne.events_from_annotations(raw, verbose=False)
event_id = {k: event_map[k] for k in ("T1", "T2") if k in event_map}

epochs = mne.Epochs(raw, events, event_id=event_id,
                    tmin=-0.5, tmax=4.0,
                    baseline=None,
                    preload=True, verbose=False)

print(epochs)
print(f"T1 trials: {sum(epochs.events[:, 2] == event_map['T1'])}")
print(f"T2 trials: {sum(epochs.events[:, 2] == event_map['T2'])}")
print(f"Total: {len(epochs)}")

<Epochs | 15 events (all good), -0.5 – 4 s (baseline off), ~4.3 MiB, data loaded,
 'T1': 8
 'T2': 7>
T1 trials: 8
T2 trials: 7
Total: 15
